In [1]:
# Step 1 - Data Loading
import pandas as pd
import numpy as np
import glob
import os

INPUT_DIR = "/kaggle/input"
csv_paths = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.csv"), recursive=True))

if not csv_paths:
    raise FileNotFoundError("No CSVs found — check you added the competition via 'Add Input'.")

male_data = {}
female_data = {}
shared_data = {}

for path in csv_paths:
    name = os.path.splitext(os.path.basename(path))[0]
    df = pd.read_csv(path)
    if name.startswith("M"):
        male_data[name] = df
    elif name.startswith("W"):
        female_data[name] = df
    else:
        shared_data[name] = df

print(f"Male files: {len(male_data)}  |  Female files: {len(female_data)}  |  Shared: {len(shared_data)}")
for label, d in [("MALE", male_data), ("FEMALE", female_data), ("SHARED", shared_data)]:
    print(f"\n{label}:")
    for name, df in sorted(d.items()):
        print(f"  {name:40s} {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols")

# Convenience references
m_teams, m_seeds = male_data.get("MTeams"), male_data.get("MNCAATourneySeeds")
m_reg_compact = male_data.get("MRegularSeasonCompactResults")
m_tourney_compact = male_data.get("MNCAATourneyCompactResults")

w_teams, w_seeds = female_data.get("WTeams"), female_data.get("WNCAATourneySeeds")
w_reg_compact = female_data.get("WRegularSeasonCompactResults")
w_tourney_compact = female_data.get("WNCAATourneyCompactResults")

sample_sub1 = shared_data.get("SampleSubmissionStage1")
sample_sub2 = shared_data.get("SampleSubmissionStage2")

Male files: 17  |  Female files: 14  |  Shared: 4

MALE:
  MConferenceTourneyGames                    7,093 rows x  5 cols
  MGameCities                               92,438 rows x  6 cols
  MMasseyOrdinals                          5,865,001 rows x  5 cols
  MNCAATourneyCompactResults                 2,585 rows x  8 cols
  MNCAATourneyDetailedResults                1,449 rows x 34 cols
  MNCAATourneySeedRoundSlots                   776 rows x  5 cols
  MNCAATourneySeeds                          2,694 rows x  3 cols
  MNCAATourneySlots                          2,653 rows x  4 cols
  MRegularSeasonCompactResults             198,577 rows x  8 cols
  MRegularSeasonDetailedResults            124,529 rows x 34 cols
  MSeasons                                      42 rows x  6 cols
  MSecondaryTourneyCompactResults            1,865 rows x  9 cols
  MSecondaryTourneyTeams                     1,895 rows x  3 cols
  MTeamCoaches                              13,900 rows x  5 cols
  MTeamConference

In [2]:
# Step 2 - Data Cleaning

def clean_seeds(seeds_df):
    df = seeds_df.copy()
    df['SeedInt'] = df['Seed'].str[1:3].astype(int)
    df['Region'] = df['Seed'].str[0]
    df['IsPlayIn'] = df['Seed'].str.len() > 3
    return df

m_seeds_clean = clean_seeds(m_seeds)
w_seeds_clean = clean_seeds(w_seeds)

def clean_results(results_df, teams_df, label=""):
    df = results_df.copy()
    loc_map = {'H': 1, 'A': -1, 'N': 0}
    df = df[df['WLoc'].isin(loc_map)]
    df['WLocNum'] = df['WLoc'].map(loc_map)
    df = df.drop_duplicates()

    valid_ids = set(teams_df['TeamID'])
    bad = ~df['WTeamID'].isin(valid_ids) | ~df['LTeamID'].isin(valid_ids)
    if bad.any():
        print(f"[{label}] dropping {bad.sum()} rows with unknown TeamID")
        df = df[~bad]
    return df

m_reg_clean = clean_results(m_reg_compact, m_teams, "M reg season")
w_reg_clean = clean_results(w_reg_compact, w_teams, "W reg season")
m_tourney_clean = clean_results(m_tourney_compact, m_teams, "M tourney")
w_tourney_clean = clean_results(w_tourney_compact, w_teams, "W tourney")

In [3]:
# Step 3 - Feature Engineering

def build_season_win_pct(reg_results_df):
    wins = reg_results_df.groupby(['Season','WTeamID']).size().rename('Wins')
    losses = reg_results_df.groupby(['Season','LTeamID']).size().rename('Losses')
    stats = pd.concat([wins, losses], axis=1).fillna(0).reset_index()
    stats.columns = ['Season','TeamID','Wins','Losses']
    stats['WinPct'] = stats['Wins'] / (stats['Wins'] + stats['Losses']).replace(0, np.nan)
    return stats

def build_training_set(tourney_df, seeds_df, winpct_df):
    seed_lookup = seeds_df.set_index(['Season','TeamID'])['SeedInt']
    wp_lookup = winpct_df.set_index(['Season','TeamID'])['WinPct']
    rows = []
    for _, g in tourney_df.iterrows():
        season, t1, t2 = g['Season'], g['WTeamID'], g['LTeamID']
        try:
            s1, s2 = seed_lookup[(season, t1)], seed_lookup[(season, t2)]
            wp1, wp2 = wp_lookup[(season, t1)], wp_lookup[(season, t2)]
        except KeyError:
            continue
        rows.append({'Season': season, 'SeedDiff': s1 - s2, 'WinPctDiff': wp1 - wp2, 'Result': 1})
        rows.append({'Season': season, 'SeedDiff': s2 - s1, 'WinPctDiff': wp2 - wp1, 'Result': 0})
    return pd.DataFrame(rows).dropna()

m_winpct = build_season_win_pct(m_reg_clean)
w_winpct = build_season_win_pct(w_reg_clean)
m_train = build_training_set(m_tourney_clean, m_seeds_clean, m_winpct)
w_train = build_training_set(w_tourney_clean, w_seeds_clean, w_winpct)

print("Men's training rows:", len(m_train), " | Women's training rows:", len(w_train))

Men's training rows: 5170  | Women's training rows: 3434


In [4]:
# Step 4 - Base Model

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, brier_score_loss

def evaluate_forward(train_df, first_val_season=2015, label=""):
    feats = ['SeedDiff', 'WinPctDiff']
    out = []
    for s in sorted(train_df['Season'].unique()):
        if s < first_val_season:
            continue
        tr = train_df[train_df['Season'] < s]     # only the past
        va = train_df[train_df['Season'] == s]    # the season we pretend not to know
        m = LogisticRegression(fit_intercept=False).fit(tr[feats], tr['Result'])
        p = m.predict_proba(va[feats])[:, 1]
        out.append({'Season': s,
                    'Brier': brier_score_loss(va['Result'], p),
                    'Brier_const_0.5': 0.25,
                    'Games': len(va) // 2})
    res = pd.DataFrame(out)
    print(f"[{label}] mean Brier = {res['Brier'].mean():.4f} over {len(res)} seasons")
    return res

m_model = evaluate_forward(m_train, label="Men's")
w_model = evaluate_forward(w_train, label="Women's")

[Men's] mean Brier = 0.1955 over 10 seasons
[Women's] mean Brier = 0.1485 over 10 seasons


In [6]:
# Step 5 - Prediction

def predict_submission(sample_sub_df, model, seeds_df, winpct_df, label=""):
    parts = sample_sub_df['ID'].str.split('_', expand=True).astype(int)
    parts.columns = ['Season','Team1','Team2']
    seed_lookup = seeds_df.set_index(['Season','TeamID'])['SeedInt']
    wp_lookup = winpct_df.set_index(['Season','TeamID'])['WinPct']

    seed_diff, winpct_diff = [], []
    for _, row in parts.iterrows():
        season, t1, t2 = row['Season'], row['Team1'], row['Team2']
        try:
            seed_diff.append(seed_lookup[(season, t1)] - seed_lookup[(season, t2)])
            winpct_diff.append(wp_lookup[(season, t1)] - wp_lookup[(season, t2)])
        except KeyError:
            seed_diff.append(np.nan)
            winpct_diff.append(np.nan)

    X_sub = pd.DataFrame({'SeedDiff': seed_diff, 'WinPctDiff': winpct_diff})
    fallback = X_sub.isna().any(axis=1)
    X_sub = X_sub.fillna(0)
    preds = np.where(fallback, 0.5, model.predict_proba(X_sub)[:, 1])
    out = sample_sub_df.copy()
    out['Pred'] = preds
    return out

In [8]:
# Step 6 - Results

# Split the sample submission by gender using TeamID ranges (men 1000s, women 3000s)
parts_check = sample_sub1['ID'].str.split('_', expand=True).astype(int)
parts_check.columns = ['Season','Team1','Team2']

is_male = parts_check['Team1'] < 2000
sub_male = sample_sub1[is_male]
sub_female = sample_sub1[~is_male]

print(f"Male matchups: {len(sub_male)}  |  Female matchups: {len(sub_female)}")

# Generate predictions for each
m_preds = predict_submission(sub_male, m_model, m_seeds_clean, m_winpct, "Men's")
w_preds = predict_submission(sub_female, w_model, w_seeds_clean, w_winpct, "Women's")

all_preds = pd.concat([m_preds, w_preds], ignore_index=True)

print("\nSample predictions:")
print(all_preds.head(10))

print("\nPrediction distribution:")
print(all_preds['Pred'].describe())

Male matchups: 261013  |  Female matchups: 258131


AttributeError: 'DataFrame' object has no attribute 'predict_proba'